# Publication Bias Analysis
## Funnel Plot · Egger Test · Trim and Fill
---
**Metodi implementati:**
- Effetto combinato a effetti fissi (inverse variance weighting)
- Funnel plot con cono al 95%
- Test di Egger (OLS manuale)
- Trim and Fill — Duval & Tweedie (2000)

> *Valori verificati contro R `meta` package: intercetta=0.8119, p=0.6124*

In [ ]:
# Install e import
!pip install numpy scipy matplotlib pandas -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from IPython.display import display

print('Tutti i pacchetti caricati')

## 1. Dati e Parametri

In [ ]:
# Dati di input — modifica qui per usare i tuoi dati reali
studi = pd.DataFrame({
    'studio':   ['Studio A', 'Studio B', 'Studio C', 'Studio D', 'Studio E'],
    'effetto':  [0.5, 0.8, 0.3, 0.6, 0.4],
    'varianza': [0.04, 0.09, 0.06, 0.16, 0.10]
})

studi['se']        = np.sqrt(studi['varianza'])
studi['peso']      = 1 / studi['varianza']
studi['precision'] = 1 / studi['se']
studi['eff_se']    = studi['effetto'] / studi['se']

eff_comb = np.average(studi['effetto'], weights=studi['peso'])
var_comb = 1 / studi['peso'].sum()
ic_orig  = [eff_comb - 1.96*np.sqrt(var_comb),
            eff_comb + 1.96*np.sqrt(var_comb)]

print(f'Studi inclusi       : {len(studi)}')
print(f'Effetto combinato   : {eff_comb:.4f}')
print(f'IC 95%              : [{ic_orig[0]:.4f}, {ic_orig[1]:.4f}]')
display(studi.round(4))

## 2. Test di Egger
Regressione OLS: `effect/SE ~ a + b*(1/SE)`
- **H0**: intercetta `a = 0` nessun bias
- **a != 0** (p < 0.05) asimmetria significativa

In [ ]:
xE    = studi['precision'].values
yE    = studi['eff_se'].values
n     = len(studi)
xMean = xE.mean()
yMean = yE.mean()

slope  = np.sum((xE - xMean)*(yE - yMean)) / np.sum((xE - xMean)**2)
interc = yMean - slope * xMean

yHat   = interc + slope * xE
resid  = yE - yHat
s2     = np.sum(resid**2) / (n - 2)
sxx    = np.sum((xE - xMean)**2)
se_int = np.sqrt(s2 * (1/n + xMean**2/sxx))
se_sl  = np.sqrt(s2 / sxx)
t_stat = interc / se_int
p_val  = 2 * (1 - stats.t.cdf(abs(t_stat), df=n-2))
r2     = 1 - np.sum(resid**2)/np.sum((yE - yMean)**2)

egger_df = pd.DataFrame({
    'Parametro': ['Intercetta (bias index)', 'SE intercetta',
                  f't-statistic (df={n-2})', 'p-value',
                  'Pendenza (true effect)', 'R2'],
    'Valore': [round(interc,4), round(se_int,4), round(t_stat,4),
               round(p_val,4), round(slope,4), round(r2,4)]
})

display(egger_df)
label = 'p >= 0.05: nessuna evidenza di publication bias' if p_val >= 0.05 \
        else 'p < 0.05: asimmetria significativa'
print(f'Interpretazione: {label}')
if n < 10:
    print(f'ATTENZIONE: k={n} < 10, potenza statistica bassa (come avverte R)')

## 3. Trim and Fill
Algoritmo iterativo Duval & Tweedie (2000):
1. **Trim**: stima k0 studi mancanti
2. **Fill**: imputa studi speculari
3. **Ricalcola** effetto combinato

In [ ]:
def trim_and_fill(eff, var, max_iter=50):
    eff = np.array(eff, dtype=float)
    var = np.array(var, dtype=float)
    n0  = len(eff)
    k0, k0_prev = 0, -1
    itr = 0

    while k0 != k0_prev and itr < max_iter:
        itr    += 1
        k0_prev = k0
        idx     = np.argsort(eff)
        eff_s   = eff[idx]
        var_s   = var[idx]
        w_s     = 1/var_s
        eff_t   = eff_s[:-k0] if k0 > 0 else eff_s
        w_t     = w_s[:-k0]   if k0 > 0 else w_s
        theta   = np.average(eff_t, weights=w_t)
        ranks   = stats.rankdata(np.abs(eff - theta))
        T_n     = np.sum(ranks * np.sign(eff - theta))
        k0      = max(0, round((4*T_n - n0*(n0+1)/2) / (2*n0 - 1)))

    idx     = np.argsort(eff)
    eff_s   = eff[idx]
    var_s   = var[idx]
    theta_f = np.average(eff_s, weights=1/var_s)
    imp_eff = np.array([2*theta_f - eff_s[-(i+1)] for i in range(k0)])
    imp_var = np.array([var_s[-(i+1)] for i in range(k0)])
    eff_aug = np.concatenate([eff, imp_eff])
    var_aug = np.concatenate([var, imp_var])
    w_aug   = 1/var_aug
    eff_c   = np.average(eff_aug, weights=w_aug)
    var_c   = 1/w_aug.sum()
    ic_c    = [eff_c - 1.96*np.sqrt(var_c), eff_c + 1.96*np.sqrt(var_c)]
    return k0, imp_eff, imp_var, eff_c, var_c, ic_c

k0, imp_eff, imp_var, eff_corr, var_corr, ic_corr = trim_and_fill(
    studi['effetto'].values, studi['varianza'].values)

tf_df = pd.DataFrame({
    'Parametro': ['Studi imputati (k0)', 'Effetto originale', 'IC 95% originale',
                  'Effetto corretto', 'IC 95% corretto', 'Delta effetto'],
    'Valore': [k0, round(eff_comb,4), f'[{ic_orig[0]:.4f}, {ic_orig[1]:.4f}]',
               round(eff_corr,4), f'[{ic_corr[0]:.4f}, {ic_corr[1]:.4f}]',
               round(eff_corr - eff_comb, 4)]
})
display(tf_df)
rob = 'Effetto robusto' if abs(eff_corr-eff_comb) < 0.05 else 'Effetto modificato'
print(f'Interpretazione: {rob}')

## 4. Grafici Publication-Ready

In [ ]:
BLUE   = '#4472C4'
RED    = '#C0392B'
ORANGE = '#E67E22'
GRAY   = '#7F8C8D'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.grid': True,
    'grid.color': '#EEEEEE',
    'grid.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

fig = plt.figure(figsize=(15, 11), facecolor='white')
fig.suptitle('Publication Bias Analysis', fontsize=17, fontweight='bold', y=0.98)
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, :])

# --- Funnel Plot ---
se_grid = np.linspace(0, max(studi['se'])*1.2, 400)
ax1.plot(eff_comb - 1.96*se_grid, se_grid, '--', color=GRAY, lw=1.3)
ax1.plot(eff_comb + 1.96*se_grid, se_grid, '--', color=GRAY, lw=1.3)
ax1.axvline(eff_comb, color=BLUE, lw=1.8, alpha=0.75,
            label=f'Effect = {eff_comb:.3f}')
ax1.scatter(studi['effetto'], studi['se'], color=RED, s=90, zorder=5)
for _, row in studi.iterrows():
    ax1.annotate(row['studio'], xy=(row['effetto'], row['se']),
                 xytext=(6,4), textcoords='offset points', fontsize=8.5, color=GRAY)
ax1.invert_yaxis()
ax1.set_xlim(-0.3, 1.3)
ax1.set_xlabel('Effect Size (d)', fontweight='bold')
ax1.set_ylabel('Standard Error', fontweight='bold')
ax1.set_title('Funnel Plot', fontweight='bold', fontsize=13)
ax1.legend(fontsize=8)

# --- Egger Regression ---
x_line  = np.linspace(min(xE)*0.85, max(xE)*1.15, 200)
y_line  = interc + slope * x_line
se_pred = np.sqrt(s2 * (1/n + (x_line - xMean)**2 / sxx))
t_crit  = stats.t.ppf(0.975, df=n-2)
ax2.fill_between(x_line, y_line - t_crit*se_pred, y_line + t_crit*se_pred,
                 color=BLUE, alpha=0.15, label='95% CI band')
ax2.plot(x_line, y_line, color=BLUE, lw=2)
ax2.axhline(0, color=RED, lw=1.2, linestyle='--', alpha=0.6)
ax2.scatter(xE, yE, color=ORANGE, s=90, zorder=5)
for _, row in studi.iterrows():
    ax2.annotate(row['studio'], xy=(row['precision'], row['eff_se']),
                 xytext=(6,4), textcoords='offset points', fontsize=8.5, color=GRAY)
ax2.set_xlabel('Precision (1/SE)', fontweight='bold')
ax2.set_ylabel('Effect / SE', fontweight='bold')
ax2.set_title(f'Egger Regression\nIntercept={interc:.4f} | p={p_val:.4f}',
              fontweight='bold', fontsize=12)

# --- Trim and Fill ---
se_grid2 = np.linspace(0, max(studi['se'])*1.2, 400)
ax3.plot(eff_corr - 1.96*se_grid2, se_grid2, '--', color=GRAY, lw=1.3)
ax3.plot(eff_corr + 1.96*se_grid2, se_grid2, '--', color=GRAY, lw=1.3)
ax3.axvline(eff_comb,  color=GRAY, lw=1.2, linestyle='dotted',
            label=f'Original effect = {eff_comb:.4f}')
ax3.axvline(eff_corr, color=BLUE, lw=1.8, alpha=0.75,
            label=f'Corrected effect = {eff_corr:.4f}')
ax3.scatter(studi['effetto'], studi['se'], color=RED, s=90,
            zorder=5, label='Original studies')
if k0 > 0:
    imp_se = np.sqrt(imp_var)
    ax3.scatter(imp_eff, imp_se, color=BLUE, s=90, marker='^',
                zorder=5, label=f'Imputed studies (k0={k0})')
    for i in range(k0):
        ax3.annotate(f'Imputed {i+1}', xy=(imp_eff[i], imp_se[i]),
                     xytext=(6,4), textcoords='offset points', fontsize=8.5, color=BLUE)
for _, row in studi.iterrows():
    ax3.annotate(row['studio'], xy=(row['effetto'], row['se']),
                 xytext=(6,4), textcoords='offset points', fontsize=8.5, color=GRAY)
ax3.invert_yaxis()
ax3.set_xlim(-0.5, 1.4)
ax3.set_xlabel('Effect Size (d)', fontweight='bold')
ax3.set_ylabel('Standard Error', fontweight='bold')
ax3.set_title(
    f'Funnel Plot - Trim and Fill | k0={k0} | '
    f'Corrected effect={eff_corr:.4f} [{ic_corr[0]:.4f}, {ic_corr[1]:.4f}]',
    fontweight='bold', fontsize=12)
ax3.legend(fontsize=9, loc='upper right')

plt.savefig('publication_bias.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('publication_bias.pdf', bbox_inches='tight', facecolor='white')
plt.show()
print('Salvato: publication_bias.png / .pdf')

## 5. Riepilogo Finale

In [ ]:
summary = pd.DataFrame({
    'Analisi': [
        'Fixed Effect — Stima', 'Fixed Effect — IC 95%',
        'Egger — Intercetta', 'Egger — p-value', 'Egger — Conclusione',
        'Trim&Fill — k0', 'Trim&Fill — Effetto corretto', 'Trim&Fill — Conclusione'
    ],
    'Risultato': [
        f'{eff_comb:.4f}',
        f'[{ic_orig[0]:.4f}, {ic_orig[1]:.4f}]',
        f'{interc:.4f}',
        f'{p_val:.4f}',
        'Nessun bias significativo' if p_val >= 0.05 else 'Bias significativo',
        str(k0),
        f'{eff_corr:.4f}',
        'Effetto robusto' if abs(eff_corr-eff_comb) < 0.05 else 'Effetto modificato'
    ]
})
print('RIEPILOGO FINALE')
display(summary)
print('Valori verificati contro R meta package')